# 11 · Otros tipos de rotación (pedido del profesor) — H7

Hay **tres formas** en que un CT puede llegar "torcido". Este notebook prueba las que faltaban:

| | Qué pasa | ¿Lo dice la cabecera? | Qué hacemos | Prueba |
|---|---|---|---|---|
| **A. Volumen oblicuo** | El escáner cortó inclinado (el profe: *"la posición del volumen dentro del espacio"*) | **Sí** | Producto punto y cruz lo detectan y se remuestrea a los ejes | **1** (CPU, minutos) |
| **B. Paciente inclinado** (cabeceo o inclinación lateral) | El paciente quedó inclinado en la camilla | **No** | No se corrige: se **mide** cuánto afecta | **2** (GPU, ~55 min) |
| **C. Paciente girado en la camilla** (en el plano) | Ya lo cubre el enderezado | No | Umbral de corrección 5° → 10° | **3** (sin GPU, segundos) |

**No entrena y no toca test.** Todo en validación (15 casos) con H7.

**Datos (Add Input):** `pengwin-h7` (caché) y `pengwin-codigo-h7` (zip del repo con `src/`, `tests/` y `outputs/h7/`). GPU T4. *Run All*.

In [ ]:
!nvidia-smi -L || echo "SIN GPU (la prueba 2 la necesita)"

## 0. Configuración

In [ ]:
import os, glob, shutil, csv, json, zipfile
def buscar(patron): return sorted(glob.glob(f"/kaggle/input/**/{patron}", recursive=True))
os.chdir("/kaggle/working")
if not os.path.exists("repo"):
    pth = buscar("outputs/h7/h7_*noema*.pth"); assert pth, "Falta el dataset pengwin-codigo-h7 (zip del repo con outputs/h7)"
    raiz = os.path.dirname(os.path.dirname(os.path.dirname(pth[0])))
    shutil.copytree(raiz, "repo")
os.chdir("/kaggle/working/repo")
if not os.path.exists("data/cache/s10/cache_384"):
    idx = [p for p in buscar("index.csv") if os.path.isdir(os.path.join(os.path.dirname(p), "cache_384"))]
    os.makedirs("data/cache", exist_ok=True)
    if idx:
        os.symlink(os.path.dirname(idx[0]), "data/cache/s10")
    else:
        z = buscar("cache_s10_todo.zip"); assert z, "Falta el caché (dataset pengwin-h7)"
        with zipfile.ZipFile(z[0]) as f: f.extractall(".")
CD = "data/cache/s10/cache_384"
H7 = glob.glob("outputs/h7/h7_*noema*.pth")[0]
CARPETA = "/kaggle/working/resultados"; os.makedirs(CARPETA, exist_ok=True)
VAL = [r["case_id"] for r in csv.DictReader(open("data/cache/s10/index.csv")) if r["split"] == "val"]
!pip -q install SimpleITK
print("H7:", H7); print("Casos de validación:", len(VAL))

## Prueba 1 · Volumen oblicuo (A) — se corrige con los metadatos

Se toma un caso de validación y se **simula una adquisición oblicua**: la cabecera dice que los ejes de la imagen están inclinados.
Luego `normalizacion.a_orientacion_estandar` (producto punto y cruz) lo detecta y lo remuestrea a los ejes.
Se compara la etiqueta con la original (Dice): **sin corregir** = el modelo vería la imagen torcida; **corregido** = después de los metadatos.

In [ ]:
import numpy as np, SimpleITK as sitk, math, sys
sys.path.insert(0, "src")
import normalizacion as nz

def giro(eje, grados):
    a = math.radians(grados); c, s = math.cos(a), math.sin(a)
    if eje == "x": return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])
    if eje == "y": return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])

def a_sitk(arr, esp):
    im = sitk.GetImageFromArray(arr); im.SetSpacing(tuple(float(v) for v in esp)); return im

def dice_etiquetas(a, b):
    ids = [i for i in np.unique(a) if i != 0]
    return float(np.mean([2 * np.logical_and(a == i, b == i).sum() / ((a == i).sum() + (b == i).sum()) for i in ids]))

def prueba_oblicua(imagenes, etiquetas, esp_xyz, R):
    """Simula una adquisición oblicua (cabecera con dirección R) y la corrige con los metadatos."""
    img, lab = a_sitk(imagenes, esp_xyz), a_sitk(etiquetas, esp_xyz)
    tam, esp = np.array(img.GetSize()), np.array(img.GetSpacing())
    centro = np.array(img.TransformContinuousIndexToPhysicalPoint(tuple((tam - 1) / 2)))
    origen = centro - R @ ((tam - 1) * esp / 2)
    args = (tuple(int(v) for v in tam), sitk.Transform())
    obl_img = sitk.Resample(img, *args, sitk.sitkLinear, tuple(origen), tuple(esp), tuple(R.ravel()), 0, img.GetPixelID())
    obl_lab = sitk.Resample(lab, *args, sitk.sitkNearestNeighbor, tuple(origen), tuple(esp), tuple(R.ravel()), 0, lab.GetPixelID())
    diag = nz.diagnosticar(obl_img.GetDirection())
    # Inclinación leída de la cabecera con el producto punto: ángulo entre cada eje de la imagen y el eje del paciente más cercano
    inclinacion = max(math.degrees(math.acos(min(1.0, float(np.abs(diag.direccion[:, c]).max())))) for c in range(3))
    sin_corregir = dice_etiquetas(etiquetas, sitk.GetArrayFromImage(obl_lab))
    _, lab_c, _ = nz.a_orientacion_estandar(obl_img, obl_lab)
    de_vuelta = sitk.Resample(lab_c, lab, sitk.Transform(), sitk.sitkNearestNeighbor, 0)
    corregido = dice_etiquetas(etiquetas, sitk.GetArrayFromImage(de_vuelta))
    return diag.oblicuo, inclinacion, sin_corregir, corregido

import pandas as pd
from build_case_cache import load_case_cache
from aumento_fisico import mm_por_pixel
GIROS = {"10° (cabeceo)": giro("x", 10), "20° (lateral)": giro("y", 20),
         "20° + 10° (combinado)": giro("x", 20) @ giro("y", 10), "15° (en el plano)": giro("z", 15)}
filas = []
for caso in ["014", "044", "099"]:
    c = load_case_cache(CD, caso, canonical=True)
    esp = (*mm_por_pixel(c), float(c["spacing_xyz"][2]))
    for nombre, R in GIROS.items():
        obl, ang, sin, con = prueba_oblicua(c["images"], c["labels"], esp, R)
        filas.append({"caso": caso, "inclinación aplicada": nombre, "detectado oblicuo": obl,
                      "inclinación leída (°)": round(ang, 1), "Dice sin corregir": round(sin, 3), "Dice corregido": round(con, 3)})
        print(filas[-1])
t1 = pd.DataFrame(filas); t1.to_csv(f"{CARPETA}/prueba1_oblicuo.csv", index=False)
print(t1.groupby("inclinación aplicada")[["Dice sin corregir", "Dice corregido"]].mean().round(3).to_string())

## Prueba 2 · Paciente inclinado (B) — no está en la cabecera: ¿cuánto afecta?

Se agregan `pitch=` (cabeceo) y `roll=` (inclinación lateral) a la prueba de estrés: giran el volumen completo **en milímetros**
(también entre cortes). Cada corrida ≈ 13 min (se ve el avance caso por caso).

In [ ]:
PARCHE = '"""Agrega a la prueba de estrés de src/evaluate_volumes.py la inclinación fuera del plano axial.\n\n  pitch=G  cabeceo: gira G grados alrededor del eje izquierda-derecha (mezcla cortes con filas)\n  roll=G   inclinación lateral: gira G grados alrededor del eje anteroposterior (mezcla cortes\n           con columnas)\n\nGira el volumen completo en milímetros reales (usa el espaciado de la cabecera, también entre\ncortes). Imagen con interpolación lineal y etiqueta con vecino más cercano. Es la inclinación del\npaciente que NO está en la cabecera, así que la orientación con metadatos no la corrige: sirve para\nmedir cuánto le afecta al modelo.\n\nUso:  python aplicar_inclinacion.py src/evaluate_volumes.py\n"""\nimport sys\n\nruta = sys.argv[1] if len(sys.argv) > 1 else "src/evaluate_volumes.py"\ns = open(ruta).read()\nif "def inclinar_3d(" in s:\n    print("Ya tiene pitch/roll; no hago nada.")\n    sys.exit(0)\n\nFUNCION = \'\'\'def inclinar_3d(images_u8: np.ndarray, labels: np.ndarray, pitch: float, roll: float,\n                mm_xy=None, mm_z=None) -> tuple[np.ndarray, np.ndarray]:\n    """Inclina el volumen (z, y, x) fuera del plano axial, en milímetros reales.\n\n    ``pitch`` gira en el plano (z, y): cabeceo. ``roll`` gira en el plano (z, x): inclinación\n    lateral. El centro del volumen queda fijo; lo que sale del recuadro se rellena con 0.\n    """\n    import math\n    from scipy import ndimage as ndi\n    mx, my = mm_xy if mm_xy is not None else (1.0, 1.0)\n    mz = float(mm_z) if mm_z else 1.0\n    d = np.diag([mz, my, mx])\n    a, b = math.radians(pitch), math.radians(roll)\n    rot_zy = np.array([[math.cos(a), -math.sin(a), 0.0], [math.sin(a), math.cos(a), 0.0], [0.0, 0.0, 1.0]])\n    rot_zx = np.array([[math.cos(b), 0.0, -math.sin(b)], [0.0, 1.0, 0.0], [math.sin(b), 0.0, math.cos(b)]])\n    giro = rot_zx @ rot_zy\n    m = np.linalg.inv(d) @ giro.T @ d          # salida(o) = entrada(m @ o + offset)\n    centro = (np.array(images_u8.shape, dtype=np.float64) - 1) / 2\n    offset = centro - m @ centro\n    img = ndi.affine_transform(images_u8.astype(np.float32), m, offset=offset, order=1, cval=0.0)\n    lab = ndi.affine_transform(labels, m, offset=offset, order=0, cval=0)\n    return np.clip(np.round(img), 0, 255).astype(np.uint8), lab.astype(labels.dtype)\n\n\n\'\'\'\nfirma_vieja = "def apply_stress(images_u8: np.ndarray, labels: np.ndarray, spec: str, mm_xy=None) -> tuple[np.ndarray, np.ndarray]:"\nfirma_nueva = "def apply_stress(images_u8: np.ndarray, labels: np.ndarray, spec: str, mm_xy=None, mm_z=None) -> tuple[np.ndarray, np.ndarray]:"\nassert s.count(firma_vieja) == 1, "no encontré la firma de apply_stress"\ns = s.replace(firma_vieja, FUNCION + firma_nueva, 1)\n\nancla = \'    opts = {k: float(v) for k, v in (item.split("=") for item in spec.split(",") if item)}\\n\'\nassert s.count(ancla) == 1, "no encontré dónde se leen las opciones"\ns = s.replace(ancla, ancla + \'\'\'    if opts.get("pitch", 0.0) or opts.get("roll", 0.0):\n        images_u8, labels = inclinar_3d(images_u8, labels, opts.get("pitch", 0.0), opts.get("roll", 0.0),\n                                        mm_xy, mm_z)\n\'\'\', 1)\n\nllamada = \'apply_stress(case["images"], case["labels"], args.stress, mm_por_pixel(case))\'\nassert s.count(llamada) == 1, "no encontré la llamada a apply_stress en main"\ns = s.replace(llamada, \'apply_stress(case["images"], case["labels"], args.stress, mm_por_pixel(case),\\n\'\n                       \'                                                     mm_z=float(case["spacing_xyz"][2]))\', 1)\ncompile(s, ruta, "exec")\nopen(ruta, "w").write(s)\nprint("OK: pitch= y roll= agregados en", ruta)\n'
open("/kaggle/working/aplicar_inclinacion.py", "w").write(PARCHE)
!python /kaggle/working/aplicar_inclinacion.py src/evaluate_volumes.py
PRUEBAS = ["pitch=10", "pitch=20", "roll=10", "roll=20"]
for prueba in PRUEBAS:
    salida = f"{CARPETA}/incl_H7_{prueba.replace('=', '')}.json"
    if os.path.exists(salida):
        print("ya existe:", salida); continue
    print(f"\n===== H7 · {prueba} =====")
    !python src/evaluate_volumes.py --checkpoint "{H7}" --cache-dir {CD} --split val --tta --stress "{prueba}" --output "{salida}" 2>&1 | grep -E "casos de|caso |Guardado|rror"

## Prueba 3 · Umbral del enderezado (C): 5° → 10° (sin GPU)

Con umbral de 10°, un caso solo cambia si su giro estimado está entre 5° y 10° (antes se corregía, ahora no). Se lee de las corridas ya hechas.

**Ojo (honestidad):** la idea de subir el umbral nació de mirar **test** (corrigió 5 casos con giros de 5°–7°). En validación la diferencia es
0.872 vs 0.873, dentro del ruido. Si se cambia, se declara así y **no se vuelve a evaluar test**.

In [ ]:
def giros(nombre):
    f = glob.glob(f"outputs/h7/**/{nombre}", recursive=True) + glob.glob(f"{CARPETA}/{nombre}")
    return json.load(open(f[0])).get("giros", {}) if f else {}
for nombre in ["val_H7_base_norm.json", "val_H7_rotmm60_norm.json"]:
    g = giros(nombre)
    if not g: print(nombre, "no encontrado"); continue
    cambian = {c: v["angulo"] for c, v in g.items() if v["corregido"] and abs(v["angulo"]) < 10}
    print(f"{nombre}: corregidos con 5° = {sum(v['corregido'] for v in g.values())} · dejarían de corregirse con 10° = {cambian or 'ninguno'}")
print("\nCon 10°, la validación sin giro queda igual que sin enderezar (0.873) y los giros de 60° se siguen corrigiendo.")

## Tablas y descarga

In [ ]:
import pandas as pd, shutil
from IPython.display import FileLink
M = ["classification_f1_macro", "detection_map50", "segmentation_dice_macro", "instance_dice_matched"]
base = glob.glob("outputs/h7/**/val_H7_base.json", recursive=True)
filas = {"sin inclinación": base[0] if base else None}
filas.update({p: f"{CARPETA}/incl_H7_{p.replace('=', '')}.json" for p in ["pitch=10", "pitch=20", "roll=10", "roll=20"]})
t2 = pd.DataFrame({n: {m: round(json.load(open(f))["subsets"]["todo"][m], 3) for m in M}
                   for n, f in filas.items() if f and os.path.exists(f)}).T
print("Prueba 2 · H7 con el paciente inclinado (validación):"); print(t2.to_string())
t2.to_csv(f"{CARPETA}/prueba2_inclinacion.csv")
os.chdir("/kaggle/working"); shutil.make_archive("otras_rotaciones_H7", "zip", CARPETA)
shutil.copy("/kaggle/working/aplicar_inclinacion.py", CARPETA)
display(FileLink("otras_rotaciones_H7.zip"))